# Phase 1: does fine-tuning on our own data beat the shipped pipeline?

Runs `backend/scripts/own_data_finetune.py` on a Colab T4. The design is pre-declared in `own_data_finetune_results.md` and committed; nothing here may change in response to the numbers.

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells in order. **Step 6 scores both locked holdouts: run it once.** The script refuses a second run into the same output folder.

In [ ]:
# 1. Confirm the GPU
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Point at the uploaded folder and check it is all there
ROOT = "/content/drive/MyDrive/own_data_finetune"
!ls -la "{ROOT}/backend/scripts" "{ROOT}/data"

In [ ]:
# 4. Install dependencies (torch is Colab's own CUDA build; this normally installs nothing)
!pip install -q -r "{ROOT}/backend/requirements-colab.txt"

In [ ]:
# 5a. Pre-flight only: checks both locked holdouts and the training file hash to
#     the values pinned when this was pre-declared, and that the shipped
#     baseline files cover exactly the same essays with exactly the same
#     labels. Trains nothing. Expect: PRE-FLIGHT PASSED
!python "{ROOT}/backend/scripts/own_data_finetune.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output" --dry-run

In [ ]:
# 5b. Smoke test: one learning rate, one epoch, a slice of the data, a 4-essay
#     slice of each holdout. Exercises every code path cheaply. Its numbers
#     mean nothing — it writes to a separate folder and never touches the
#     real holdout scoring.
!python "{ROOT}/backend/scripts/own_data_finetune.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output_smoke" --smoke

In [ ]:
# 6. THE RUN — run once. Trains the rubric-aware model, scores both locked
#    holdouts, measures CPU inference cost. Expect roughly 10–15 minutes:
#    one model x two learning rates x 4 epochs on ~100 short training rows.
!python "{ROOT}/backend/scripts/own_data_finetune.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output"

In [ ]:
# 7. Show the headline numbers, then flush Drive so nothing is lost
import json
r = json.load(open(f"{ROOT}/output/own_data_finetune.json"))

print("chosen:", r["fit"]["chosen"], "| overfit signal:", r["fit"]["any_overfit_signal"])
print()
for holdout_name in ("holdout2", "essay2_holdout"):
    for criterion, block in r[holdout_name].items():
        ft, sh = block["fine_tuned"], block["shipped"]
        print(f"{holdout_name:16} {criterion:9} fine-tuned MAE {ft['mae']:.2f} QWK {ft['qwk']:.2f}"
              f"   shipped MAE {sh['mae']:.2f} QWK {sh['qwk']:.2f}")
        c = block["comparisons"]["mae"]
        mark = "distinguishable" if c["distinguishable"] else "NOT distinguishable"
        print(f"{'':16} {'':9} fine-tuned - shipped MAE {c['finetuned_minus_shipped']:+.2f}"
              f" [{c['ci'][0]:+.2f}, {c['ci'][1]:+.2f}] -- {mark}")
print()
print(f"{r['cost']['parameters']/1e6:.1f}M params, "
      f"{r['cost'].get('cpu_ms_per_submission_mean', float('nan')):.0f} ms/submission on CPU, "
      f"{r['cost'].get('saved_model_mb', float('nan')):.0f} MB saved")

drive.flush_and_unmount()
print("\nDone. Download output/own_data_finetune.json and output/own_data_finetune_log.txt.")